# From an activated conda instance open jupyter notebook to Analyze Xenium data

In [ ]:
# import sys
# !{sys.executable} -m pip install matplotlib seaborn scanpy squidpy anndata harmonypy leidenalg -q

In [ ]:
import os, shutil, zipfile, tempfile, warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq
import anndata as ad
import harmonypy as hm
import hisepy as hp
from spatialdata_io import xenium

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

Download the Feature-cell Matrix (HDF5) and the Cell summary file (CSV).
You need these 2 files in a new folder:
- `cell_feature_matrix.h5`
- `cells.csv`

(note the cells.csv file is compressed and accessible via `gunzip filename.csv.gz`)

In [ ]:
DATA_DIR = '/home/workspace/PCD_data/xenium/data'

OUTS_DIR = '/home/workspace/PCD_data/xenium/data/outs'
os.makedirs(OUTS_DIR, exist_ok=True)

FIGURES_DIR = '../figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
sc.settings.figdir = FIGURES_DIR
plt.rcParams['savefig.dpi'] = 300

COMBINED_ADATA_DIR = "/home/workspace/PCD_data/xenium/data/combined_clustered.h5ad"
combined = sc.read_h5ad(COMBINED_ADATA_DIR)

In [ ]:
XENIUM_SAMPLES = {
    # name (→ OUTS_DIR/<name>/)   HISE UUID                               section   disease
    'TIS05393':     '70f7c0d6-8d7e-4b86-acf3-ca80137fd849',  # 009     Multiple Myeloma
    'TIS04788':     '38fe47a0-c09f-4373-9a8a-8a957c40aa9f',  # 010     Multiple Myeloma
    'TIS05392':     '0c879d26-970f-47fc-a705-88b103b4de80',  # 007     Multiple Myeloma
    'TIS05394':     'd7b4b14e-9856-4169-a363-213b4e29a99e',  # 008     Multiple Myeloma
    'TIS04789':     '7c3113c3-23bf-4840-8f45-61505048322c',  # 017     Multiple Myeloma
    'TIS05397_008': '16752455-bd23-4f88-8d05-2108e7778496',  # 008     Normal BM (2024)
}

def download_xenium(name, uuid, outs_dir):
    dest = os.path.join(outs_dir, name)
    if os.path.isdir(dest):
        print(f'{name}: already extracted — skipping')
        return

    print(f'{name}: downloading from HISE (UUID {uuid})…')
    (zip_path,) = hp.cache_files(file_ids=[uuid])

    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)

        # The ZIP unpacks to one top-level output folder; move it to dest.
        inner = next(
            os.path.join(tmp, d) for d in os.listdir(tmp)
            if os.path.isdir(os.path.join(tmp, d))
        )
        shutil.move(inner, dest)

    print(f'{name}: extracted → {dest}')


for name, uuid in XENIUM_SAMPLES.items():
    download_xenium(name, uuid, OUTS_DIR)

def read_xenium(name, outs_dir):
    path = os.path.join(outs_dir, name)
    sdata = xenium(path)
    adata = sdata.tables['table']
    # spatialdata_io populates obsm['spatial'] directly; no centroid columns in obs.
    return adata


sk1 = read_xenium('TIS05393', OUTS_DIR)
sk2 = read_xenium('TIS04788', OUTS_DIR)
sk3 = read_xenium('TIS05392', OUTS_DIR)
sk4 = read_xenium('TIS05394', OUTS_DIR)
sk5 = read_xenium('TIS04789', OUTS_DIR)
sk6 = read_xenium('TIS05397_008', OUTS_DIR)

for name, adata in [
    ('TIS05393', sk1), ('TIS04788', sk2), ('TIS05392', sk3),
    ('TIS05394', sk4), ('TIS04789', sk5), ('TIS05397_008', sk6),
]:
    print(f'{name}: {adata.n_obs:,} cells, {adata.n_vars} genes')

sk1.obs_names_make_unique()
sk2.obs_names_make_unique()
sk3.obs_names_make_unique()
sk4.obs_names_make_unique()
sk5.obs_names_make_unique()
sk6.obs_names_make_unique()

combined = sc.concat(
    {'TIS05393': sk1, 'TIS04788': sk2, 'TIS05392': sk3,
     'TIS05394': sk4, 'TIS04789': sk5, 'TIS05397_008': sk6},
    label='sample',
    join='inner',
)
combined.obs_names_make_unique()

print(f'{combined.n_obs:,} cells x {combined.n_vars} genes')
print(combined.obs['sample'].value_counts())

In [ ]:
print(f'Number of observations (cells): {combined.n_obs}')
print(f'Number of variables (genes): {combined.n_vars}')

In [ ]:
import matplotlib as mpl
import scipy.sparse as sp

combined.layers["counts"] = combined.X.copy()
sc.pp.normalize_total(combined, inplace=True)
# sc.pp.log1p(combined)  # Uncomment to log-transform

# Ensure X stays sparse after normalization
if not sp.issparse(combined.X):
    combined.X = sp.csr_matrix(combined.X)

In [ ]:
samples_dict = {
    'TIS05393': sk1, 'TIS04788': sk2, 'TIS05392': sk3,
    'TIS05394': sk4, 'TIS04789': sk5, 'TIS05397_008': sk6,
}

# n_genes_by_counts: genes with at least one count per cell
for adata in samples_dict.values():
    adata.obs['n_genes_by_counts'] = np.asarray((adata.X > 0).sum(axis=1)).flatten()

# ── Per-sample distributions ──────────────────────────────────────────────────
fig, axes = plt.subplots(len(samples_dict), 3, figsize=(15, 3.2 * len(samples_dict)), squeeze=False)

for row_axes, (name, adata) in zip(axes, samples_dict.items()):
    counts = adata.obs['total_counts'].to_numpy()
    genes  = adata.obs['n_genes_by_counts'].to_numpy()

    row_axes[0].hist(np.log10(counts + 1), bins=60, color='steelblue')
    row_axes[0].set_xlabel('log10(counts + 1)')
    row_axes[0].set_title(f'{name} — counts  (median {np.median(counts):.0f})')

    row_axes[1].hist(np.log10(genes + 1), bins=60, color='steelblue')
    row_axes[1].set_xlabel('log10(genes + 1)')
    row_axes[1].set_title(f'{name} — genes  (median {np.median(genes):.0f})')

    row_axes[2].plot(np.arange(1, len(counts) + 1), np.sort(counts)[::-1], lw=1.5)
    row_axes[2].set_xscale('log')
    row_axes[2].set_yscale('log')
    row_axes[2].set_xlabel('cell rank')
    row_axes[2].set_ylabel('counts')
    row_axes[2].set_title(f'{name} — knee')

fig.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'qc_distributions.{ext}'), dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    name: {
        'n_cells':       adata.n_obs,
        'median_counts': int(np.median(adata.obs['total_counts'])),
        'median_genes':  int(np.median(adata.obs['n_genes_by_counts'])),
    }
    for name, adata in samples_dict.items()
}).T
print(summary.to_string())

In [ ]:
QC_COLUMNS = ['total_counts', 'n_genes_by_counts']

fig, axes = plt.subplots(len(samples_dict), len(QC_COLUMNS),
                         figsize=(5.5 * len(QC_COLUMNS), 5 * len(samples_dict)), squeeze=False)

for i, (name, adata) in enumerate(samples_dict.items()):
    for j, column in enumerate(QC_COLUMNS):
        sc.pl.spatial(
            adata, color=column, vmax='p99', spot_size=50,
            ax=axes[i, j], show=False,
            title=f'{name} — {column}  (median {adata.obs[column].median():.0f})',
        )

fig.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'spatial_qc_maps.{ext}'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# percent_top default is (50, 100, 200, 500), but 500 > 377 genes in the Xenium panel.
sc.pp.calculate_qc_metrics(combined, percent_top=None, inplace=True)

# ── Cell filter ───────────────────────────────────────────────────────────────
# Adjust these based on the histograms above: look for the dip between the two
# humps (if present) or the knee of the rank curve.
MIN_COUNTS = 15
MIN_GENES  = 10

n_before = combined.n_obs
sc.pp.filter_cells(combined, min_counts=MIN_COUNTS)
sc.pp.filter_cells(combined, min_genes=MIN_GENES)
n_after = combined.n_obs

print(f'Filtered {n_before - n_after:,} cells  '
      f'({100 * (n_before - n_after) / n_before:.1f}% removed)')
print(f'Remaining: {n_after:,} cells')
print(combined.obs['sample'].value_counts())

In [ ]:
sc.pp.pca(combined, n_comps=30, use_highly_variable=False, random_state=0)
sc.pp.neighbors(combined, n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.umap(combined, random_state=0)
sc.tl.leiden(combined, key_added='leiden_prebatch', random_state=0)

In [ ]:
sc.pl.umap(
    combined,
    color=["total_counts", "n_genes_by_counts", "leiden_prebatch"],
    wspace=0.4,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'umap_prebatch_UMAP_MM.{ext}'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Harmony batch correction (bypassing scanpy wrapper due to version mismatch)
import harmonypy as hm

ho = hm.run_harmony(
    combined.obsm['X_pca'].astype('float64'),
    combined.obs,
    'sample',
    random_state=0,
)

# Handle shape depending on harmonypy version (Z_corr may be n_pcs x n_cells or n_cells x n_pcs)
z = ho.Z_corr
if z.shape[0] != combined.n_obs:
    z = z.T
combined.obsm['X_pca_harmony'] = z
combined.obsm['X_pca'] = combined.obsm['X_pca_harmony']

sc.pp.neighbors(combined, use_rep='X_pca', random_state=0)
sc.tl.umap(combined, random_state=0)

In [ ]:
sc.pl.umap(
    combined,
    color=["total_counts", "n_genes_by_counts", 'sample'],
    wspace=0.2,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'umap_harmony_UMAP_MM.{ext}'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
RESOLUTIONS = [round(0.1 * i, 2) for i in range(1, 16)]

DEG_SWEEP_DIR = os.path.join(DATA_DIR, 'resolution_sweep')
os.makedirs(DEG_SWEEP_DIR, exist_ok=True)

sweep_keys = []
for res in RESOLUTIONS:
    key = f'leiden_res{res:.2f}'
    sweep_keys.append(key)

    sc.tl.leiden(combined, flavor='igraph', n_iterations=2, directed=False,
                 resolution=res, key_added=key, random_state=0)

    n_clusters = combined.obs[key].nunique()
    print(f'resolution {res:.2f}: {n_clusters} clusters')

    sc.tl.rank_genes_groups(combined, key, method='wilcoxon',
                            key_added=f'rank_{key}')
    deg = sc.get.rank_genes_groups_df(combined, group=None, key=f'rank_{key}')
    deg.to_csv(os.path.join(DEG_SWEEP_DIR, f'deg_{key}_wilcoxon.csv'), index=False)

print(f'\nWrote {len(RESOLUTIONS)} DEG tables to {DEG_SWEEP_DIR}')

In [ ]:
out_path = os.path.join(DATA_DIR, 'combined_clustered.h5ad')
combined.write_h5ad(out_path)

In [ ]:
sc.pl.umap(combined, color=sweep_keys, wspace=0.05, ncols=3, show=False, legend_loc='on data', size=10)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'umap_resolution_sweep.{ext}'),
                dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
RESOLUTIONS = [round(0.1 * i, 2) for i in range(1, 16)]
sweep_keys  = [f'leiden_res{res:.2f}' for res in RESOLUTIONS]

transitions = []
for lo, hi in zip(sweep_keys[:-1], sweep_keys[1:]):
    ct = pd.crosstab(combined.obs[lo], combined.obs[hi])
    transitions.append(ct)

def evenly_spaced(labels):
    n = len(labels)
    return {lab: i / (n - 1) if n > 1 else 0.5 for i, lab in enumerate(labels)}

labels_0 = sorted(combined.obs[sweep_keys[0]].unique(), key=int)
x_pos = [evenly_spaced(labels_0)]

for i, ct in enumerate(transitions):
    child_labels = sorted(ct.columns.tolist(), key=int)
    parent_x = x_pos[i]
    raw_x = {}
    for lab in child_labels:
        col   = ct[lab]
        total = col.sum()
        wx    = sum(parent_x.get(str(p), 0.5) * v for p, v in col.items() if v > 0)
        raw_x[lab] = wx / total if total > 0 else 0.5
    ordered = sorted(child_labels, key=lambda l: raw_x[l])
    x_pos.append(evenly_spaced(ordered))

# Cell counts per cluster per resolution.
n_cells = [{str(k): v for k, v in combined.obs[key].value_counts().items()} for key in sweep_keys]
max_n   = max(n for level in n_cells for n in level.values())

# Draw — edges first so nodes sit on top.
fig, ax = plt.subplots(figsize=(16, 10))
y = np.linspace(0, 1, len(sweep_keys))

for i, ct in enumerate(transitions):
    max_flow = ct.values.max()
    for lab_l in ct.index:
        for lab_h in ct.columns:
            flow = ct.loc[lab_l, lab_h]
            if flow == 0:
                continue
            frac = flow / max_flow
            ax.plot(
                [x_pos[i][str(lab_l)], x_pos[i + 1][str(lab_h)]],
                [y[i], y[i + 1]],
                color='#3D6DB5', lw=7 * frac,
                alpha=0.2 + 0.6 * frac,
                solid_capstyle='round', zorder=1,
            )

for i, (xd, nd) in enumerate(zip(x_pos, n_cells)):
    for lab, x_val in xd.items():
        n  = nd.get(str(lab), 1)
        sz = 120 + 1400 * (n / max_n)
        ax.scatter(x_val, y[i], s=sz, color='#3D6DB5',
                   edgecolors='white', linewidths=1.5, zorder=3)
        ax.text(x_val, y[i], str(lab),
                ha='center', va='center', fontsize=7,
                color='white', fontweight='bold', zorder=4)

ax.set_yticks(y)
ax.set_yticklabels(
    [f'res {res:.1f}   ({len(x_pos[i])} clusters)' for i, res in enumerate(RESOLUTIONS)],
    fontsize=10,
)
ax.set_xticks([])
ax.set_xlim(-0.08, 1.08)
ax.set_ylim(-0.06, 1.06)
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title('Clustree — Leiden cluster stability across resolutions', fontsize=13, pad=12)
fig.tight_layout()

for ext in ['png', 'pdf']:
    fig.savefig(os.path.join(FIGURES_DIR, f'clustree.{ext}'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
KEY = 'leiden_res0.60'

cluster_to_cell_type_res06 = {
    "0":  "Erythroid Precursor 1",
    "1":  "Erythroid Precursor 2",
    "2":  "Neoplastic Plasma Cell 1 (MYC/CXCR4-high)",
    "3":  "Neoplastic Plasma Cell 2 (LILRB4-high)",
    "4":  "Endothelial",
    "5":  "Stromal Fibroblast",
    "6":  "Neoplastic Plasma Cell 3",
    "7":  "Cytotoxic T Cell",
    "8":  "Macrophage",
    "9":  "Monocyte",
    "10": "Proliferating Myeloid Precursor",
}

combined.obs['cell_type_res06'] = (
    combined.obs[KEY].map(cluster_to_cell_type_res06).astype('category')
)

plt.rcParams['savefig.dpi'] = 300
sc.pl.umap(
    combined,
    color='cell_type_res06',
    legend_loc='on data',
    # legend_loc='right margin',
    legend_fontsize=8,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'UMAP_celltypes_res06.{ext}'),
                bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
KEY = 'leiden_res0.80'

cluster_to_cell_type_res08 = {
    "0":  "Erythroid Precursor 1",
    "1":  "Erythroid Precursor 2",
    "2":  "Endothelial",
    "3":  "Neoplastic Plasma Cell 1",
    "4":  "Neoplastic Plasma Cell 2 (BCL2L11-high, apoptosis-primed)",
    "5":  "Neoplastic Plasma Cell 3",
    "6":  "TREM2+ Macrophage (Tumor-Supportive, Adjacent)",
    "7":  "Monocyte",
    "8":  "Neoplastic Plasma Cell 4",
    "9":  "Macrophage",
    "10": "Cytotoxic T Cell",
    "11": "Neoplastic Plasma Cell 5 (MYC-high, possibly proliferative)",
    "12": "Plasmablast (B-to-Plasma Cell Transitional)",
    "13": "Stromal Fibroblast",
    "14": "Neoplastic Plasma Cell 6",
    "15": "Lymphocyte/Dendritic Cell/Vessel (Intravascular)",
    "16": "Proliferating Myeloid Precursor",
    "17": "Lipid-Laden Macrophage (Tumor-Adjacent)",
}

combined.obs['cell_type_res08'] = (
    combined.obs[KEY].map(cluster_to_cell_type_res08).astype('category')
)

plt.rcParams['savefig.dpi'] = 300
sc.pl.umap(
    combined,
    color='cell_type_res08',
    legend_loc='on data',
    # legend_loc='right margin',
    legend_fontsize=8,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'UMAP_celltypes_res08.{ext}'),
                bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
plt.rcParams['savefig.dpi'] = 300

sc.pl.umap(
    combined,
    color=["total_counts", "n_genes_by_counts", "leiden", 'sample'],
    wspace=0.4,
    show=False,
)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'umap_leiden_UMAP_MM.{ext}'), dpi=300, bbox_inches='tight')
plt.show()

sc.tl.rank_genes_groups(combined, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(combined, n_genes=10, sharey=False, fontsize=20, show=False)
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'rank_genes_groups.{ext}'), dpi=300, bbox_inches='tight')
plt.show()

# Persist immediately so the cluster labels used in the next cell's
# cluster_to_cell_type mapping are frozen and don't silently drift if
# any cell above gets re-run out of order or the notebook is restarted.
combined.write('combined_clustered.h5ad')

In [ ]:
df = sc.get.rank_genes_groups_df(combined, group=None)
df.to_csv(os.path.join(DATA_DIR, 'deg_leiden_wilcoxon.csv'), index=False)

In [ ]:
# IMPORTANT: this mapping is manually curated by reading the marker genes in the
# rank_genes_groups plot from the cell above. It is only valid for the clustering
# that produced 'combined_clustered.h5ad'. If cell 11 is re-run (even with
# the fixed random_state, e.g. after changing input data, package versions, or
# resolution) always regenerate this mapping by re-reading the new marker-gene
# plot -- do not assume cluster numbers stay pinned to the same cell types.
cluster_to_cell_type = {
    "0":  "Erythroid Precursor 1",
    "1":  "Erythroid Precursor 2",
    "2":  "Endothelial",
    "3":  "Neoplastic Plasma Cell 1",
    "4":  "Neoplastic Plasma Cell 2 (BCL2L11-high, apoptosis-primed)",
    "5":  "Neoplastic Plasma Cell 3",
    "6":  "TREM2+ Macrophage (Tumor-Supportive, Adjacent)",
    "7":  "Monocyte",
    "8":  "Neoplastic Plasma Cell 4",
    "9":  "Macrophage",
    "10": "Cytotoxic T Cell",
    "11": "Neoplastic Plasma Cell 5 (MYC-high, possibly proliferative)",
    "12": "Plasmablast (B-to-Plasma Cell Transitional)",
    "13": "Stromal Fibroblast",
    "14": "Neoplastic Plasma Cell 6",
    "15": "Lymphocyte/Dendritic Cell/Vessel (Intravascular)",
    "16": "Proliferating Myeloid Precursor",
    "17": "Lipid-Laden Macrophage (Tumor-Adjacent)",
}

combined.obs['cell_type'] = combined.obs['leiden'].map(cluster_to_cell_type)

In [ ]:
combined.obs['cell_type'] = combined.obs['cell_type'].astype('category')
print(combined.obs['cell_type'].cat.categories)

In [ ]:
plt.rcParams['savefig.dpi'] = 300

sc.pl.umap(
    combined,
    color='cell_type',
    legend_loc='right margin',
    legend_fontsize=8,
    show=False,
)

for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'UMAP_celltypes.{ext}'), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
slide_ids = combined.obs['sample'].unique()
genes_to_plot = ['MZB1', 'PECAM1', 'GNLY']

cmap = mpl.colors.ListedColormap(['cyan', 'magenta', 'orange', 'red']).with_extremes(under='0.8', over='red')
bounds = [3, 4, 8, 20]
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

fig, axes = plt.subplots(len(slide_ids), len(genes_to_plot), figsize=(5 * len(genes_to_plot), 5 * len(slide_ids)))

for i, slide_id in enumerate(slide_ids):
    slide_data = combined[combined.obs['sample'] == slide_id]
    for j, gene in enumerate(genes_to_plot):
        ax = axes[i, j] if len(slide_ids) > 1 else axes[j]
        sc.pl.spatial(slide_data, color=gene, ax=ax, spot_size=50, cmap=cmap, norm=norm, show=False)
        if i == 0:
            ax.set_title(gene)
        if j == 0:
            ax.set_ylabel(f'Slide {slide_id}')

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(os.path.join(sc.settings.figdir, f'spatial_gene_expression.{ext}'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def split_umap(combined, split_by, ncol=2, nrow=None, save_stem=None, **kwargs):
    categories = combined.obs[split_by].cat.categories
    if nrow is None:
        nrow = int(np.ceil(len(categories) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(5*ncol, 4*nrow), dpi=300)
    axs = axs.flatten()
    for i, cat in enumerate(categories):
        ax = axs[i]
        sq.pl.spatial_scatter(
            combined[combined.obs[split_by] == cat],
            library_id="spatial",
            shape=None,
            color=["leiden"],
            wspace=0.4,
            ax=ax,
            title=cat
        )
    plt.tight_layout()
    if save_stem is not None:
        for ext in ['png', 'pdf']:
            plt.savefig(f'{save_stem}.{ext}', format=ext, dpi=300, bbox_inches='tight')
    plt.show()


split_umap(combined=combined, split_by='sample', ncol=3,
           save_stem=os.path.join(sc.settings.figdir, 'spatial_leiden_by_sample'))